# COM428 - Generative Modeling (week 7)
##### notebook code (C) 2023-2025 Timothy James Becker
## ITSL CH10: Neural Network (NN) Methods 
#### (with descrimanative model examples)

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/6/63/Typical_cnn.png/593px-Typical_cnn.png" width=600>

Artificial Neural Networks have been around for a very long time (1960s...) and have enjoyed several cycles of popularity followed by abandonment (for SVM, Random Forest, etc) only to become more popular then ever given new forms (Recurrent RNN, Convolutional CNN, Deep refers to the number of connected layers).  The first effective form was the Multilayer Perceptron which is a full-connected layer and had some success in the 1990s.  We will get into the various forms of networks (topology) along with the application areas in which they excel. We will use the [keras](https://keras.io) NN framework that is part of the [tensorflow](https://tensorflow.rstudio.com/installation/) package.

In [7]:
import tensorflow as tf
import keras
from keras import layers
print(tf.__version__)
print(keras.__version__)
print("Num CPUs Available: ", len(tf.config.list_physical_devices('CPU'))) 
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU'))) #check to see if your GPU is working...

2.18.1
3.6.0
Num CPUs Available:  1
Num GPUs Available:  0


### 10.1-10.2 Single/Multiple Layer Networks

NN are similiar to Trees in some ways since they approximate linear or non-linear functions.  They differ in the construction of the network which is variable in topology and has a great impact on performance and learning. In general it is easier to balance and fit NNs than tree emthods such as Random Forest and NNs work well with high dimension data which other methods have difficultly with such as images, text, etc. [Interactive NN](http://playground.tensorflow.org). Mathematically a simply feed-forward network FFNN can be expressed as:

#### $f(X) = \beta_0 + \displaystyle\sum_{k=1}^{K}\beta_kh_k(X) = \beta_0 + \sum_{k=1}^{K}\beta_kg(w_{k0}+\sum_{j=1}^{p}w_{k,j}X_j)$ $\quad\quad$(10.1)

The $K$ is a selectable number of hidden units in the single layer with $p$ input features (input data)

#### $A_k = h_x(X)=g(w_{k0} + \displaystyle\sum_{j=1}^{p} w_{k,j}X_j)$ $\quad\quad$ (10.2)

$g(z)$ above is called an activation function which is fixed across the layer (in advance) and each $A_k$ is like a transformation on the original $X_1, ..., X_P$ features.

When you feed the $K$ activations from the hidden layer into the output you have:

#### $f(X) = \beta_0 + \displaystyle\sum_{k=1}^{K}\beta_kA_k$ $\quad\quad$ (10.3)

which is esential a linear regression model on the $K$ activations.  

Some well known and often used non-linear activation functions for NN are:

#### <u>Sigmoid</u>: 

#### $g(z) = \frac{1}{1+(\frac{1}{e})^z} = (1+e^{-z})^{-1}$ $\quad\quad$ (10.4)

#### <u>Rectified Linear Unit (ReLU)</u>

#### $g(z) = \begin{cases}
    0       & \quad \text{if } z<0 \\
    z  & \quad \text{if } z>=0
  \end{cases}$ $\quad\quad$ (10.5)
Activations are thought of as firing (passing on values) when the threshold for activation is passed (can think of like 1.0)

<br>
<br>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/Colored_neural_network.svg/420px-Colored_neural_network.svg.png" width=400>

#### <u>Question</u>: Why don't we use a linear function for $g(z)$?

#### [EX]

Given $p=2$, $K=2$ and $g(z)=|z|^{\frac{1}{3}}$ and parameters: 

$\beta_0=0,\quad \beta_1=1,\quad \beta_2=-1$

$w_{1,0}=0,\quad w_{1,1}=1,\quad w_{1,2}=1$

$w_{2,0}=0,\quad w_{2,1}=1,\quad w_{2,2}=-1$

using (10.2) we have:

$A_1=h_1(X)= g(w_{k0} + \displaystyle\sum_{j=1}^{p} w_{k,j}X_j) = g(0 + 1X_1 + 1X_2) = |X_1 + X_2|^{\frac{1}{3}}$

$A_2=h_2(X)= |0 + 1X_1 - 1X_2| = |X_1 - X_2|^{\frac{1}{3}}$

using (10.7 and 10.1) we then get:

$f(X) = \beta_0 + \displaystyle\sum_{k=1}^{K}\beta_kh_k(X) = 0 + 1A_1 - A_2 = |X_1+X_2|^{\frac{1}{3}} - |X_1-X_2|^{\frac{1}{3}}$

now plot this function:

In [9]:
import matplotlib.pyplot as plt
import numpy as np
fig = plt.figure();ax = fig.add_subplot(1, 1, 1)
ax.spines['bottom'].set_position('zero');ax.spines['left'].set_position('zero')
ax.xaxis.set_ticks_position('bottom');ax.yaxis.set_ticks_position('right')

X1 = np.linspace(1,100)
X2 = np.linspace(-100,100)
f1 = abs(X1+X2)**(1/3)-abs(X1-X2)**(1/3)
plt.plot(X1,f1,label='$f(X)=|X_1+X_2|^{1/3}-|X_1-X_2|^{1/3}$')
plt.legend();plt.show()

ModuleNotFoundError: No module named 'matplotlib'

For multilayer networks (MLP) we feed the output of the activation into the next layer instead of the original input. An so you get to select (have to) $K_L$ for $L$ layers of the NN.

#### <u>Multiclass Using Soft-Max</u>

When we want to do 3+-classification tasks using NN models, we can make use of the soft-max method:

#### $f_m(X) = Pr(Y=m | X) = \frac{e^{Z_m}}{\sum_{l=0}^{L-1}e^{Z_l}}$

Here $Z_m$ is a linear model for each class from the final output

#### Building NNs in keras
keras is a library for expressing NN model archatectures, paramters, activation function along with many other features and functionalities. [keras](https://keras.io) also incorporates back-ends for accelerating the training/inference process which is often very slow on CPU (since it is matrix based).

In [10]:
#from https://keras.io building a very basic MLP model
mlp = keras.Sequential(name="mlp_model")
mlp.add(layers.Input((16,)))
mlp.add(layers.Dense(32,activation="sigmoid", name="dense_layer_1"))
mlp.add(layers.Dense(32,activation="sigmoid", name="dense_layer_2"))
mlp.add(layers.Flatten(name="flatten_1"))                              #do we need a flatten when all dim is 1D?
mlp.add(layers.Dense(5,activation="softmax",name="final_out"))
mlp.compile(loss='sparse_categorical_crossentropy',
            optimizer='adam',
            metrics=['sparse_categorical_accuracy'])
mlp.build()
mlp.summary()

Model: "mlp_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_layer_1 (Dense)           │ (None, 32)             │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_layer_2 (Dense)           │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_out (Dense)               │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,765 (6.89 KB)

 Trainable params: 1,765 (6.89 KB)

 Non-trainable params: 0 (0.00 B)

#### Machine Learning?
In keras the optimizer is the alogrithm used to try different values for all the model parameters. The loss function is a positive valued function that is minimized by the optimizer. In general you write your loss function to penalize incorrect or predictions that are far away from what you want (really bad guesses). The metric is the function that provides the basic premise for evaluating (in human terms) how good a model is. Generally metrics such as accuracy will be between 0.0 and 1.0 which tralsates to 0% to 100% (getting no predictions correct verus getting all predictions correct)

In [11]:
import numpy as np

label_map = {1:0,10:1,100:2,200:3,300:4}
map_label = {label_map[k]:k for k in label_map}

n = 8       #this will be the number of random vectors/rows: AKA observations
X,Y = [],[] #this is the data and the base dist
for i in range(n):
    Y += [np.random.choice(list(label_map))]        #randomly pick one of these as lambda
    X += [np.random.poisson(Y[-1],16)]              #make a vector of size 16 from that dist
X = np.asarray(X,dtype=int)                         #reform N so that is in one np object/array
Y = np.asarray([label_map[y] for y in Y],dtype=int)
print(X)
print(Y)

[[199 177 200 234 198 232 193 203 179 207 190 201 196 197 205 203]
 [100  78 114  89  88  97  91 101  90  89  78  89 104 111 111 112]
 [  2   0   0   1   3   0   3   1   1   0   1   0   1   0   0   1]
 [315 316 285 307 326 323 274 282 309 281 291 309 286 303 291 280]
 [  1   3   2   0   0   1   1   4   3   1   2   0   2   4   2   1]
 [232 216 201 212 195 230 166 197 205 200 199 212 191 199 206 207]
 [207 208 207 213 195 235 200 207 200 221 190 198 201 198 217 205]
 [ 15   7  10   4   5   9  12   8  10   8   9  11  10  11  11  10]]
[3 2 0 4 0 3 3 1]


In [12]:
#mlp.fit(X,Y)
mlp.fit(X,Y,batch_size=64,epochs=10,validation_split=0.3)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - loss: 2.1645 - sparse_categorical_accuracy: 0.2000 - val_loss: 1.5622 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 2.1359 - sparse_categorical_accuracy: 0.2000 - val_loss: 1.5680 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 2.1089 - sparse_categorical_accuracy: 0.2000 - val_loss: 1.5727 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 2.0832 - sparse_categorical_accuracy: 0.2000 - val_loss: 1.5693 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 2.0554 - sparse_categorical_accuracy: 0.2000 - val_loss: 1.5664 - val_sparse_categorical_accuracy: 0.0000e+00
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 2.0231 - sparse_categorical_accuracy: 0.2000 - val_loss: 1.5658 - val_sparse_categorical_accuracy: 0.0000e+00


In [13]:
y = np.random.choice(list(label_map))
x = np.random.poisson(y,16)
x = np.asarray([x],dtype=int)
y = np.asarray([label_map[y]],dtype=int)
x,y

(array([[109, 100,  93,  99, 101,  95,  99,  95,  90,  94,  97, 101,  92,
          98,  94,  92]]),
 array([2]))

In [14]:
mlp.predict(x),y #start not very good...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


(array([[0.05967905, 0.12271216, 0.22778374, 0.25040883, 0.33941624]],
       dtype=float32),
 array([2]))

### 10.3 Convolution Layers (CNN)

Convolution nodes in a network are design to learn the features and so are a type of feature-mapping or feature learning.  They have proven to be effective at image classification tasks by making use of these hierarchical feature-maps.

If we assume a small input picture:

#### $\begin{pmatrix} a & b & c \\ d & e & f \\ g & h & i \\ j & k & l\end{pmatrix}$

And then convolve it with the convulation matrix:

#### $\begin{pmatrix} \alpha & \beta \\ \gamma & \sigma \end{pmatrix}$

We get:

#### $\begin{pmatrix} a\alpha+b\beta+d\gamma+e\sigma & b\alpha+c\beta+e\gamma+f\sigma \\  d\alpha+e\beta+g\gamma+h\sigma & e\alpha+f\beta+h\gamma+i\sigma \\ g\alpha+h\beta+j\gamma+k\sigma & h\alpha+i\beta+k\gamma+l\sigma \end{pmatrix}$

More generally convolutional kernels are odd sizes such as 3x3, 5x5, etc:

[convolutional kernels are 2D image filters](https://en.wikipedia.org/wiki/Kernel_(image_processing))

#### Pooling Convolutional Layers

Convolutional Layers for images are very large (think HD images are 1080x720 pixels is ~1 million nodes!). Pooling is a way to reduce the total node count (lower the estimated feature space) while retaining most of the underlying information content.  You can think of it as a type of convolutional down-sampling (reducing the image size)

<u>Max Pooling</u>: Take the maximal value in the matrix (2x2 here):

#### $\begin{pmatrix} 30 & 21 & 10 & 13 \\ 31 & 26 & 15 & 12 \\ 28 & 17 & 7 & 9 \\ 18 & 26 & 5 & 2\end{pmatrix}$

Would become:

#### $\begin{pmatrix} 31 & 15 \\ 28 & 9\end{pmatrix}$


#### Dropout, Batch Regularization, Kernel Regularization

There are many ways to decrease overfitting with CNN

<u>Dropout</u>: remove some of the connections between layers

<u>Batch Regularization</u> make all training batches fit within 0-1

<u>Kernel Regularization</u> creates a penalty for updating/changing node weights (in the hidden layers) this means the model will try to change the weights as little as possible



#### Max Pooling

Then there are several ways to create more robust models that can deal with shifts or geometric transofrmations. In image processing it is often called positional invariance. Two main methods can be employed here:

Maximum pooling (downsampling the dimensions from the CNN layer above it) has the effect of allowing the gradients to move and still be funcitonally detected.

A nice example can be found on kaggle [here](https://www.kaggle.com/code/ryanholbrook/maximum-pooling) but we can and should employ it in our won work and either use it or not!

#### Data Augmentation (images)

The other major way to create robust models is to perterb them using domain knowledge. In the case of mage classification the transformation or the object can play a hige role. Even a basic affine transform can make the mode more robust to orientation (seeing the same object with some rotation or scaling).  You can learn more about image transformations here: [opencv geometry](https://docs.opencv.org/4.5.5/da/d6e/tutorial_py_geometric_transformations.html)

#### CNNs are automated Image Processing

The operations that we have discussed so far are basically filters that have the abilty to learn patterns in the images.  To really understand the CNN for image tasks, we would need some more background in computer-based image processing techniques.  This is an entire area in CS which is called computer vision.  The most interesting open-source system that would proivde a great learning environment for the student is [opencv](https://docs.opencv.org).  In particuliar the section on [kernel filters](https://docs.opencv.org/4.x/d4/d13/tutorial_py_filtering.html) would give an important insight into what we are actually doing with CNNs in an image classification framework.

#### <u>CNN Picture example</u>

A classical example of when to use CNN is for images and image classification tasks.  A popular example dataset is included with the tensorflow/keras R packaged which is called [MNIST](http://yann.lecun.com/exdb/mnist/).  It is a set of small images that are hand written numbers (0-9).  We can train a CNN model to learn how to do number recognition by framing this as a classification problem.

In [15]:
#from https://keras.io/2.15/api/

#import the dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data() #load digit data set
c   = len(set(y_train))                     #will find the unique number of labels to use for final model output

# Scale images to the [0, 1] range
x_train = x_train.astype("float32")/255 #we use 255 because these are unsigned 8 bits -> 255 is max value
x_test  = x_test.astype("float32")/255

# Make sure images have shape (28, 28, 1): make them 3D!
x_train = np.expand_dims(x_train, -1)
x_test  = np.expand_dims(x_test,  -1)
dim = x_train.shape[1:]
print("x_train shape:", x_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")
print('model input dimension:',dim)

# convert class vectors to binary class matrices
y_train = keras.utils.to_categorical(y_train,c)
y_test  = keras.utils.to_categorical(y_test, c)

x_train shape: (60000, 28, 28, 1)
60000 train samples
10000 test samples
model input dimension: (28, 28, 1)


In [27]:
# make out cnn model here... from=https://keras.io/examples/vision/mnist_convnet/, https://www.tensorflow.org/tutorials/images/cnn

cnn = keras.Sequential(name="mnist_cnn_model")
cnn.add(layers.Input(dim,name='input_layer'))
cnn.add(layers.Conv2D(8, kernel_size=(3, 3), activation="relu"))
cnn.add(layers.MaxPooling2D(pool_size=(2, 2)))
cnn.add(layers.Conv2D(12, kernel_size=(3, 3), activation="relu"))
cnn.add(layers.MaxPooling2D(pool_size=(2, 2)))
cnn.add(layers.Flatten(name="flatten_2d_to_1d"))
cnn.add(layers.Dropout(0.5,name="final_drop_out"))
cnn.add(layers.Dense(c,activation="softmax",name="final_output"))
cnn.compile(loss="categorical_crossentropy",
            optimizer=keras.optimizers.SGD(learning_rate=1e-2), 
            metrics=["accuracy"])

#add more layers to make a deep CNN AKA: deep learning model

cnn.build()
cnn.summary()

Model: "mnist_cnn_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 26, 26, 8)      │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 13, 13, 8)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 11, 11, 12)     │           876 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 5, 12)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2d_to_1d (Flatten)      │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_drop_out (Dropout)        │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ final_output (Dense)            │ (None, 10)             │         3,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,966 (15.49 KB)

 Trainable params: 3,966 (15.49 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
cnn.fit(x=x_train,y=y_train,batch_size=32,epochs=2,validation_split=0.5)

Epoch 1/2
938/938 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.7814 - loss: 0.6800 - val_accuracy: 0.9241 - val_loss: 0.2913
Epoch 2/2
938/938 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.8578 - loss: 0.4705 - val_accuracy: 0.9424 - val_loss: 0.2179


In [31]:
pred_prob = cnn.predict(x_test) #this gives us the softmax probabilty estimate
y_l = np.argmax(y_test,axis=1)
p_l = np.argmax(pred_prob,axis=1)
y_l,p_l

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step


(array([7, 2, 1, ..., 4, 5, 6]), array([7, 2, 1, ..., 4, 5, 6]))

In [32]:
a = 0
for i in range(len(y_l)):
    if y_l[i]==p_l[i]: a += 1
a/len(y_l)

0.948

#### <u>Question</u> Is this performance realistic to what you would get using this model in the real-world?